In [1]:
from huggingface_hub import hf_hub_download
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
from aeon.classification.interval_based import RSTSF
from sklearn.metrics import accuracy_score
from autotsc.models import StackerV4, LokyStackerV5, LokyStackerV5SoftRF, LokyStackerV6, LokyStackerV7
from autotsc import utils
from aeon.classification.convolution_based import MultiRocketHydraClassifier
import matplotlib.pyplot as plt  
def load_monash_fold(dataset_name: str, fold: int, test_size: float = 0.2):
    path_X = hf_hub_download(
        repo_id=f"monster-monash/{dataset_name}",
        filename=f"{dataset_name}_X.npy",
        repo_type="dataset",
    )
    path_y = hf_hub_download(
        repo_id=f"monster-monash/{dataset_name}",
        filename=f"{dataset_name}_y.npy",
        repo_type="dataset",
    )
    X = np.load(path_X, mmap_mode="r")
    y = np.load(path_y, mmap_mode="r")

    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=fold)
    train_idx, test_idx = next(sss.split(X, y))

    return X[train_idx], y[train_idx], X[test_idx], y[test_idx]

X_train, y_train, X_test, y_test = load_monash_fold("LakeIce", 10)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)
X_train = X_train.astype(np.float64)
X_test = X_test.astype(np.float64)
print(X_train.dtype)
print(y_train.dtype)

(103424, 1, 161) (103424,) (25856, 1, 161) (25856,)
float64
int64


In [2]:
np.unique(y_train)

array([0, 1, 2])

In [3]:
#for i in range(50):
#    yv = y_train[i]
#    color = 'blue'
#    if yv == 1:
#        color = 'orange'
#    if yv == 2:
#        color = 'green'
#    plt.plot(X_train[i, 0, :], color=color, alpha=0.2)

In [4]:
f = 10
X_train_sub = X_train[::f]
y_train_sub = y_train[::f]
print(X_train_sub.shape, y_train_sub.shape)

(10343, 1, 161) (10343,)


In [5]:
#f = 50
X_test_sub = X_test[::f]
y_test_sub = y_test[::f]
print(X_test_sub.shape, y_test_sub.shape)

(2586, 1, 161) (2586,)


In [ ]:
m = LokyStackerV7(random_state=491, n_repetitions=1, n_jobs=16, verbose=10)
m.fit(X_train_sub, y_train_sub)
pred = m.predict(X_test_sub)
print(accuracy_score(y_test_sub, pred))
m.cleanup()

In [ ]:
m = MultiRocketHydraClassifier(random_state=491, n_jobs=16)
m.fit(X_train_sub, y_train_sub)
pred = m.predict(X_test_sub)
print(accuracy_score(y_test_sub, pred))

In [ ]:
from aeon.classification.interval_based import QUANTClassifier
m = QUANTClassifier(random_state=491)
m.fit(X_train_sub, y_train_sub)
pred = m.predict(X_test_sub)
print(accuracy_score(y_test_sub, pred))